# D2 — Root Cause Analysis (RCA) Assignment

Notebook này thực hiện pipeline RCA cho bài D2:

1. Load `cluster_summary.json`
2. Load `alerts_sample.jsonl`, `services.json`, `incidents_history.json`
3. Build service graph bằng `networkx`
4. Chạy graph + temporal scoring để tìm top-3 root cause candidates
5. Retrieve similar incidents bằng keyword/kNN-style similarity
6. Gán `class` và `actions` từ incident giống nhất
7. Ghi `results/rca_output.json`

In [7]:
from pathlib import Path
import json
from datetime import datetime
from pprint import pprint

import networkx as nx

BASE = Path.cwd()
DATASET_DIR = BASE / "dataset"
RESULTS_DIR = BASE / "results"
RESULTS_DIR.mkdir(exist_ok=True)

CLUSTER_SUMMARY_PATH = DATASET_DIR / "cluster_summary.json"
ALERTS_PATH = DATASET_DIR / "alerts_sample.jsonl"
SERVICES_PATH = DATASET_DIR / "services.json"
INCIDENTS_PATH = DATASET_DIR / "incidents_history.json"
OUTPUT_PATH = RESULTS_DIR / "rca_output.json"

print("BASE:", BASE)
print("Cluster summary path:", CLUSTER_SUMMARY_PATH)
print("Output path:", OUTPUT_PATH)

BASE: /workspaces/aiops-haikhoa-keys/w2/d2
Cluster summary path: /workspaces/aiops-haikhoa-keys/w2/d2/dataset/cluster_summary.json
Output path: /workspaces/aiops-haikhoa-keys/w2/d2/results/rca_output.json


## 1. Helper functions để load dữ liệu

In [8]:
def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_jsonl(path: Path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def parse_time(value):
    if isinstance(value, datetime):
        return value
    if value is None:
        return None
    return datetime.fromisoformat(str(value).replace("Z", "+00:00"))


def ensure_list_clusters(cluster_summary):
    """Support nhiều format cluster_summary khác nhau từ D1."""
    if isinstance(cluster_summary, list):
        return cluster_summary
    if isinstance(cluster_summary, dict):
        for key in ["clusters", "results", "cluster_summary"]:
            if key in cluster_summary and isinstance(cluster_summary[key], list):
                return cluster_summary[key]
    raise ValueError("Không tìm thấy list clusters trong cluster_summary.json")

## 2. Load input files

Nếu cell này lỗi, kiểm tra lại bạn đã đặt đúng file vào:

- `dataset/cluster_summary.json`
- `dataset/alerts_sample.jsonl`
- `dataset/services.json`
- `dataset/incidents_history.json`

In [9]:
cluster_summary = load_json(CLUSTER_SUMMARY_PATH)
alerts = load_jsonl(ALERTS_PATH)
services = load_json(SERVICES_PATH)
incidents = load_json(INCIDENTS_PATH)
clusters = ensure_list_clusters(cluster_summary)

print("clusters:", len(clusters))
print("alerts:", len(alerts))
print("services records:", len(services) if isinstance(services, list) else len(services.keys()))
print("incidents:", len(incidents))

print("\nSample cluster:")
pprint(clusters[0])

clusters: 3
alerts: 20
services records: 5
incidents: 2

Sample cluster:
{'alert_count': 17,
 'alert_ids': ['a-0001',
               'a-0002',
               'a-0003',
               'a-0004',
               'a-0005',
               'a-0006',
               'a-0007',
               'a-0008',
               'a-0009',
               'a-0011',
               'a-0012',
               'a-0014',
               'a-0015',
               'a-0016',
               'a-0017',
               'a-0018',
               'a-0020'],
 'cluster_id': 'c-000-000',
 'fingerprints': ['cart-svc|latency_p99_ms|warn',
                  'checkout-svc|downstream_payment_error_rate|crit',
                  'checkout-svc|latency_p99_ms|crit',
                  'checkout-svc|latency_p99_ms|warn',
                  'checkout-svc|request_drop_rate|crit',
                  'edge-lb|p99_latency_ms|crit',
                  'edge-lb|upstream_5xx_rate|crit',
                  'edge-lb|upstream_5xx_rate|warn',
                

## 3. Build service graph

Quy ước: cạnh `A -> B` nghĩa là `A` gọi/phụ thuộc vào `B`. Khi `B` hỏng, lỗi có thể lan ngược về `A`.

In [10]:
def normalize_services(raw_services):
    """Convert services.json về list service records."""
    if isinstance(raw_services, list):
        return raw_services
    if isinstance(raw_services, dict):
        if "services" in raw_services and isinstance(raw_services["services"], list):
            return raw_services["services"]
        # fallback: dict dạng {service_name: metadata}
        normalized = []
        for name, meta in raw_services.items():
            if isinstance(meta, dict):
                row = {"service": name, **meta}
            else:
                row = {"service": name}
            normalized.append(row)
        return normalized
    raise ValueError("services.json format không được hỗ trợ")


def get_service_name(record):
    for key in ["service", "name", "id", "service_name"]:
        if key in record:
            return record[key]
    return None


def get_dependencies(record):
    for key in ["depends_on", "dependencies", "calls", "downstream"]:
        if key in record and isinstance(record[key], list):
            return record[key]
    return []


def build_graph(raw_services):
    records = normalize_services(raw_services)
    g = nx.DiGraph()

    for record in records:
        name = get_service_name(record)
        if not name:
            continue
        g.add_node(name, **record)

    for record in records:
        name = get_service_name(record)
        if not name:
            continue
        for dep in get_dependencies(record):
            if isinstance(dep, dict):
                dep_name = get_service_name(dep)
            else:
                dep_name = dep
            if dep_name:
                g.add_edge(name, dep_name)

    return g

graph = build_graph(services)
print("nodes:", graph.number_of_nodes())
print("edges:", graph.number_of_edges())
print("sample edges:", list(graph.edges())[:10])

nodes: 10
edges: 0
sample edges: []


## 4. Graph + temporal RCA scoring

In [13]:
def alert_id(alert):
    for key in ["id", "alert_id"]:
        if key in alert:
            return alert[key]
    return None


def alert_service(alert):
    for key in ["service", "service_name"]:
        if key in alert:
            return alert[key]
    labels = alert.get("labels", {})
    if isinstance(labels, dict):
        return labels.get("service") or labels.get("service_name")
    return None


def alert_timestamp(alert):
    for key in ["timestamp", "ts", "startsAt", "start_time"]:
        if key in alert:
            return parse_time(alert[key])
    return None


def cluster_id(cluster, idx=None):
    for key in ["cluster_id", "id", "cluster"]:
        if key in cluster:
            return cluster[key]
    return f"c-{idx:03d}" if idx is not None else "unknown"


def cluster_services(cluster, cluster_alerts):
    for key in ["services", "service_names", "services_involved"]:
        if key in cluster and isinstance(cluster[key], list):
            return set(cluster[key])
    return {alert_service(a) for a in cluster_alerts if alert_service(a)}


def cluster_alert_ids(cluster):
    for key in ["alert_ids", "alerts", "members"]:
        if key in cluster and isinstance(cluster[key], list):
            values = cluster[key]
            if values and isinstance(values[0], dict):
                return {alert_id(v) for v in values}
            return set(values)
    return set()


def get_cluster_alerts(cluster, all_alerts):
    ids = cluster_alert_ids(cluster)
    if ids:
        matched = [a for a in all_alerts if alert_id(a) in ids]
        if matched:
            return matched

    # fallback nếu D1 cluster đã lưu trực tiếp alert objects
    if isinstance(cluster.get("alerts"), list) and cluster["alerts"] and isinstance(cluster["alerts"][0], dict):
        return cluster["alerts"]

    # fallback bằng service overlap
    svcs = set(cluster.get("services", []))
    return [a for a in all_alerts if alert_service(a) in svcs]


def service_first_seen(cluster_alerts):
    first_seen = {}
    for alert in cluster_alerts:
        svc = alert_service(alert)
        ts = alert_timestamp(alert)
        if not svc or not ts:
            continue
        if svc not in first_seen or ts < first_seen[svc]:
            first_seen[svc] = ts
    return first_seen


def timestamp_scores(first_seen):
    if not first_seen:
        return {}
    if len(first_seen) == 1:
        return {svc: 1.0 for svc in first_seen}

    min_t = min(first_seen.values())
    max_t = max(first_seen.values())
    span = (max_t - min_t).total_seconds()
    if span <= 0:
        return {svc: 1.0 for svc in first_seen}

    return {
        svc: round(1.0 - ((ts - min_t).total_seconds() / span), 4)
        for svc, ts in first_seen.items()
    }


def graph_temporal_candidates(cluster, cluster_alerts, graph, top_k=3):
    svcs = cluster_services(cluster, cluster_alerts)
    svcs = {svc for svc in svcs if svc}

    # Nếu service không nằm trong graph, vẫn add node để không bị mất output
    subgraph = graph.subgraph([svc for svc in svcs if svc in graph.nodes]).copy()
    for svc in svcs:
        if svc not in subgraph.nodes:
            subgraph.add_node(svc)

    if len(subgraph.nodes) == 0:
        return []

    reverse_graph = subgraph.reverse(copy=True)
    try:
        pagerank = nx.pagerank(reverse_graph, alpha=0.85)
    except Exception:
        pagerank = {svc: 1.0 for svc in svcs}

    max_pr = max(pagerank.values()) if pagerank else 1.0
    pagerank_norm = {svc: pagerank.get(svc, 0.0) / max_pr for svc in svcs}

    first_seen = service_first_seen(cluster_alerts)
    ts_score = timestamp_scores(first_seen)

    candidates = []
    for svc in svcs:
        graph_score = pagerank_norm.get(svc, 0.0)
        time_score = ts_score.get(svc, 0.5 if first_seen else 0.0)
        final_score = 0.6 * graph_score + 0.4 * time_score
        candidates.append([svc, round(float(final_score), 4)])

    candidates.sort(key=lambda x: x[1], reverse=True)
    return candidates[:top_k]

# Test thử trên cluster đầu tiên
first_cluster_alerts = get_cluster_alerts(clusters[0], alerts)
print("alerts in first cluster:", len(first_cluster_alerts))
print(graph_temporal_candidates(clusters[0], first_cluster_alerts, graph))

alerts in first cluster: 17
[['payment-svc', 1.0], ['checkout-svc', 0.9391], ['edge-lb', 0.8976]]


## 5. Retrieve similar incidents

Heuristic similarity:

- `+0.4` nếu root cause trong history nằm trong service của cluster
- `+0.2` mỗi service overlap, tối đa `+0.4`
- `+0.2` nếu cùng severity

In [15]:
def cluster_severity(cluster, cluster_alerts=None):
    for key in ["severity", "severity_max", "max_severity"]:
        if key in cluster:
            return cluster[key]
    if cluster_alerts:
        severities = [a.get("severity") or a.get("labels", {}).get("severity") for a in cluster_alerts]
        severities = [s for s in severities if s]
        if severities:
            # đơn giản: lấy severity xuất hiện đầu tiên nếu không có ranking cụ thể
            return severities[0]
    return None


def incident_id(incident):
    for key in ["incident_id", "id"]:
        if key in incident:
            return incident[key]
    return None


def incident_services(incident):
    for key in ["services_involved", "services", "service_names"]:
        if key in incident and isinstance(incident[key], list):
            return set(incident[key])
    return set()


def incident_class(incident):
    for key in ["root_cause_class", "class", "category"]:
        if key in incident:
            return incident[key]
    return "other"


def incident_actions(incident):
    for key in ["actions", "remediation", "recommended_actions"]:
        if key in incident:
            value = incident[key]
            if isinstance(value, list):
                return value
            if isinstance(value, str):
                return [value]
    return ["Investigate manually"]


def similarity(cluster, cluster_alerts, incident):
    svcs = cluster_services(cluster, cluster_alerts)
    sev = cluster_severity(cluster, cluster_alerts)
    hist_services = incident_services(incident)
    score = 0.0

    if incident.get("root_cause_service") in svcs:
        score += 0.4

    overlap = len(svcs & hist_services)
    score += min(0.4, 0.2 * overlap)

    if sev and incident.get("severity") == sev:
        score += 0.2

    return round(score, 4)


def normalize_incidents(incidents):
    if isinstance(incidents, dict):
        raw = list(incidents.values())
    else:
        raw = incidents

    result = []
    for item in raw:
        if isinstance(item, dict):
            result.append(item)
        elif isinstance(item, list):
            result.extend([x for x in item if isinstance(x, dict)])

    return result


def retrieve_similar(cluster, cluster_alerts, incidents, top_k=3):
    incident_list = normalize_incidents(incidents)

    scored = []
    for inc in incident_list:
        s = similarity(cluster, cluster_alerts, inc)
        if s >= 0.2:
            scored.append((inc, s))

    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]


def classify_from_history(similar_scored):
    if not similar_scored:
        return "other", ["Investigate manually"]
    top_incident, _score = similar_scored[0]
    return incident_class(top_incident), incident_actions(top_incident)

similar_demo = retrieve_similar(clusters[0], first_cluster_alerts, incidents)
[(incident_id(inc), score) for inc, score in similar_demo]

[('INC-2025-11-08', 0.8), ('INC-2026-03-20', 0.8), ('INC-2025-08-17', 0.6)]

## 6. Run RCA cho toàn bộ clusters và ghi output

In [16]:
ALLOWED_CLASSES = {
    "connection_pool_exhaustion", "slow_query", "memory_leak", "rebalance_storm",
    "deadlock", "network_partition", "bad_deploy", "config_push",
    "tls_expiry", "ddos", "other"
}


def validate_result(result):
    if not result.get("graph_top3"):
        result["graph_top3"] = []
    if result.get("class") not in ALLOWED_CLASSES:
        result["class"] = "other"
    if not isinstance(result.get("confidence"), (int, float)):
        result["confidence"] = 0.0
    result["confidence"] = max(0.0, min(1.0, float(result["confidence"])))
    if not isinstance(result.get("actions"), list) or not result["actions"]:
        result["actions"] = ["Investigate manually"]
    return result


def run_rca(clusters, alerts, graph, incidents):
    output = {
        "clusters_analyzed": len(clusters),
        "results": []
    }

    for idx, cluster in enumerate(clusters):
        c_alerts = get_cluster_alerts(cluster, alerts)
        top3 = graph_temporal_candidates(cluster, c_alerts, graph, top_k=3)
        similar = retrieve_similar(cluster, c_alerts, incidents, top_k=3)
        root_class, actions = classify_from_history(similar)

        root_cause = top3[0][0] if top3 else "unknown"
        confidence = top3[0][1] if top3 else 0.0

        result = {
            "cluster_id": cluster_id(cluster, idx),
            "graph_top3": top3,
            "root_cause": root_cause,
            "class": root_class,
            "confidence": confidence,
            "actions": actions,
            "reasoning": (
                f"{root_cause} is ranked top-1 by combined graph PageRank and timestamp score. "
                f"Graph score captures dependency impact; timestamp score boosts services that alerted earlier."
            ),
            "similar_incidents": [incident_id(inc) for inc, _score in similar if incident_id(inc)],
            "method": "graph+retrieval"
        }
        output["results"].append(validate_result(result))

    return output

rca_output = run_rca(clusters, alerts, graph, incidents)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(rca_output, f, indent=2, ensure_ascii=False)

print(f"Wrote: {OUTPUT_PATH}")
pprint(rca_output)

Wrote: /workspaces/aiops-haikhoa-keys/w2/d2/results/rca_output.json
{'clusters_analyzed': 3,
 'results': [{'actions': ['Rollback to v3.1. Scale pool 50 → 100 cushion. Add '
                          'pool monitor alert > 80%.'],
              'class': 'connection_pool_exhaustion',
              'cluster_id': 'c-000-000',
              'confidence': 1.0,
              'graph_top3': [['payment-svc', 1.0],
                             ['checkout-svc', 0.9391],
                             ['edge-lb', 0.8976]],
              'method': 'graph+retrieval',
              'reasoning': 'payment-svc is ranked top-1 by combined graph '
                           'PageRank and timestamp score. Graph score captures '
                           'dependency impact; timestamp score boosts services '
                           'that alerted earlier.',
              'root_cause': 'payment-svc',
              'similar_incidents': ['INC-2025-11-08',
                                    'INC-2026-03-20',
   

## 7. Acceptance check nhanh

In [17]:
assert OUTPUT_PATH.exists(), "Thiếu results/rca_output.json"
assert rca_output["clusters_analyzed"] >= 1, "Chưa phân tích cluster nào"
assert isinstance(rca_output["results"], list) and rca_output["results"], "results rỗng"

required_fields = {
    "cluster_id", "graph_top3", "root_cause", "class", "confidence",
    "actions", "reasoning", "similar_incidents", "method"
}

for row in rca_output["results"]:
    missing = required_fields - set(row.keys())
    assert not missing, f"Thiếu field: {missing}"
    assert isinstance(row["graph_top3"], list), "graph_top3 phải là list"
    assert row["class"] in ALLOWED_CLASSES, f"class không hợp lệ: {row['class']}"
    assert isinstance(row["actions"], list) and row["actions"], "actions phải là list non-empty"

print("Acceptance check passed")

Acceptance check passed


## 8. Gợi ý viết FINDINGS.md

Sau khi chạy notebook, mở `results/rca_output.json`, lấy cluster có confidence cao nhất hoặc cluster chính để viết:

- Root cause là service nào?
- Vì sao graph + timestamp chọn service đó?
- Confidence có đủ để auto-remediation không?
- Case nào bạn chưa chắc và vì sao?
- Có chọn bonus không? Nếu không, giải thích retrieval-only đã đủ.